# 5.1 Python OOPs

**Prerequisites:** 04 Functions (all notebooks)
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What OOP is, and when a class beats a bag of functions
- Classes, objects, `__init__` and `self` — built around a task-queue `Job`
- **`__repr__` vs `__str__`** — and why `__repr__` comes first
- Encapsulation done honestly: `_convention`, `__name_mangling`, `@property`
- Instance vs class variables; class methods and static methods
- All the inheritance shapes, method overriding, `super()`, and the MRO (C3)
- Composition and aggregation; abstract base classes
- Operator overloading; **`__eq__` and `__hash__` together**; `@total_ordering`
- Custom iterators; object lifecycle (`__new__`, `__del__`); **`__slots__`**
- A dunder reference table, and every meaning of the underscore

---

## Why object-oriented programming?

Every program manages two things: **data**, and **the functions that operate on that data**.
The question is only how you keep them together as the program grows.

### Procedural programming
The style you have used so far: a series of functions, each performing one task, passing data
around between them.

```python
job = {"name": "resize-image", "attempts": 0, "max_retries": 3}
record_attempt(job)
if is_exhausted(job):
    move_to_dead_letter(job)
```

Nothing is wrong with this — until the program grows. Then the cracks show:

- Nothing stops `job["atempts"] = 1` (typo: new key, silent bug) or `job["attempts"] = -7`.
- `record_attempt` *hopes* it gets a job dict; hand it a user dict and it fails somewhere else, later.
- The data (`dict`) and the functions that understand it live apart, so every caller must know
  which functions belong to which shape of dict.

### Object-oriented programming
OOP moves the functions **into** the data structure. An **object** carries its data
(**attributes**) and the operations that belong to that data (**methods**) as one unit:

```python
job = Job("resize-image", max_retries=3)
job.record_attempt()
if job.exhausted():
    ...
```

A **class** is the template that defines what every such object looks like — defining a class
creates a brand-new *type*, exactly like the built-in `str` or `list`. The class does nothing
by itself; you **instantiate** it to get objects (also called **instances**), as many as you need.

> **Analogy that earns its keep:** the class is the *database schema*, the objects are the *rows*.
> The schema defines what columns exist and their constraints; each row is one concrete record.
> You never store data "in the schema" — and you rarely store data in the class itself.

### A little history, and the payoff
OOP is not modern — the roots go back to the 1960s (**Simula 67** was the first language with
objects). It stuck because it buys, when applied with taste:

| Advantage | Meaning in practice |
|---|---|
| **Grouping** | State + behaviour travel together; no orphaned helper functions |
| **Reusability** | Inherit or compose instead of copy-paste |
| **Encapsulation** | Internals can change without breaking callers |
| **Extensibility** | New variants (new job types, new storage backends) slot in beside old ones |
| **Maintainability** | The place to look for job-related bugs is the `Job` class |

Python is not "purely" object-oriented — you are never forced to write a class, and a module of
plain functions is often the right call (more on that in the pitfalls at the end). But nearly
everything you *touch* in Python is already an object:

In [ ]:
# Everything in Python is already an object - every value carries its type (its class):
print(type(60))
print(type('GET /health'))
print(type([200, 301, 404]))
print(type(print))  # even functions are objects
# Defining a class of our own (below) adds a brand-new type to this list.

## Defining a class

The `class` keyword introduces one. Named parts:

```python
class Job:                                  # `class` keyword + name in CapWords
    """A unit of background work."""        # docstring: first statement, describes the class
    ...                                     # body: methods and class-level names go here
```

- By convention the name uses **CapWords** (`Job`, `HttpResponse`, `RetryPolicy`) — this is how
  readers tell classes from functions at a glance.
- The class body is a new **namespace**: everything defined inside belongs to the class.
- Every class you create implicitly inherits from the base class `object`.

Our running example for this notebook is a **task queue** — the kind that resizes images or
sends invoices in the background. Start with the smallest class that is still legal:

In [ ]:
class Job:
    """A unit of background work in a task queue."""   # docstring

print(Job, type(Job))

The class itself is an object too — its type is `type`. Defining `Job` added a new type to
the language, on equal footing with `int` and `str`.

### Built-in class attributes

Every class carries some bookkeeping attributes for free:

| Attribute | Contains |
|----|---|
| `__doc__` | The docstring |
| `__name__` | The class name as a string |
| `__module__` | Name of the module the class was defined in |
| `__bases__` | Tuple of base classes (`(object,)` unless you inherited) |
| `__dict__` | The class namespace, as a mapping |

⚠️ **Version note (3.13+):** the compiler now strips common leading whitespace from docstrings,
so an indented multi-line `__doc__` prints without the indentation it has in the source file.
Purely cosmetic, but it can surprise doc-generating tools that compare strings exactly.

In [ ]:
print(Job.__doc__)
print(Job.__name__)
print(Job.__module__)
print(Job.__bases__)   # every class implicitly inherits from `object`

## Objects, instances, instantiation

- **Object / instance:** one concrete value built from the class. *Instance of `Job`* and
  *`Job` object* mean the same thing.
- **Instantiation:** creating one, by *calling the class* like a function: `Job()`.

Each call returns a **separate** object:

In [ ]:
job_a = Job()   # calling the class returns a new object
job_b = Job()

print(type(job_a))
print(type(job_a) is Job)
print(job_a is job_b)     # two distinct objects, even though built identically

### Instance variables — data unique to each object

An **instance variable** (or instance attribute) holds data belonging to *one* object.
Python lets you attach them from outside at any time with plain assignment —
`object.attribute = value`:

In [ ]:
job_a.name = 'resize-image'
job_a.max_retries = 3

job_b.name = 'send-invoice'
job_b.max_retries = 5

# Two ways to see an object's own attributes as a dict:
print(job_a.__dict__)
print(vars(job_b))       # vars(obj) is the readable spelling of obj.__dict__

In [ ]:
# dir() shows MORE than vars(): everything reachable, including what
# the class inherited from `object` - which is where all these dunders come from:
print(dir(job_a))

- `vars(obj)` → only the attributes stored **on this object**.
- `dir(obj)` → everything reachable *through* the object, including names inherited from the
  class and from `object`. We declared no members in `Job`, yet `dir()` is full — that is the
  `object` inheritance showing.

Attaching attributes by hand like this works, but it is exactly the procedural-dict problem
wearing a new syntax: nothing guarantees every job *has* a `name`, or that `max_retries` is set
before someone reads it. We want every instance initialised **automatically, the same way** —
which is the constructor's job.

## `__init__` — the constructor

`__init__` is a method Python calls **automatically** on every new instance, immediately after
creating it. Whatever arguments you pass to `Job(...)` arrive in `__init__`, and its job is to
put the instance into a valid starting state.

```python
class Job:
    def __init__(self, name, max_retries=3):
        ─┬─────    ─┬──  ─┬─────────────
         │          │     └─ parameters the caller supplies: Job('resize-image', 5)
         │          └─ the new, still-empty instance (Python passes it for you)
         └─ dunder name: "initialise"

        self.name = name          # create instance variables ON the new object
        ─┬────────   ─┬──
         │            └─ local parameter (exists only during this call)
         └─ attribute stored on the instance (lives as long as the object)
```

- An `__init__` written with only `self` is a **default constructor**; with extra parameters it
  is a **parameterized constructor**. Defaults (`max_retries=3`) work like any function default.
- `__init__` initialises an object that already exists — the actual *creation* is `__new__`'s
  job, which you will meet near the end of this notebook.

In [ ]:
class Job:
    """A unit of background work in a task queue."""

    def __init__(self, name, max_retries=3):
        self.name = name
        self.max_retries = max_retries
        self.attempts = 0        # not everything must come from the caller

job_a = Job('resize-image')                    # __init__ runs: self=the new object
job_b = Job('send-invoice', max_retries=5)

print(vars(job_a))
print(vars(job_b))

### `self`

`self` is simply **the instance the method was invoked on**. It is not a keyword — the name is
pure convention (universally followed; linters flag anything else).

When you write `job_a.record_attempt()`, Python translates it to
`Job.record_attempt(job_a)` — the object slides in as the first argument. That is all `self`
is: the explicit spelling of "this object".

### Reading, changing and deleting attributes

In [ ]:
# Get:
print(job_a.name)

# Set (rebinding an existing attribute):
job_a.max_retries = 4
print(vars(job_a))

# Delete:
del job_a.attempts
print(vars(job_a))          # 'attempts' is gone from THIS object only
print(vars(job_b))          # job_b still has its own

### The attribute functions: `getattr`, `setattr`, `hasattr`, `delattr`

Dot notation needs the attribute name **written in the source code**. When the name arrives at
run time — from a config file, an API payload, a plugin registry — you need the function forms:

| Function | Task |
|----|---|
| `getattr(obj, name, default)` | Read attribute by name-string; `AttributeError` if missing, unless a `default` is given |
| `setattr(obj, name, value)` | Set attribute by name-string |
| `hasattr(obj, name)` | `True` if the attribute exists |
| `delattr(obj, name)` | Delete attribute by name-string |

This is how serialization libraries copy fields, how test frameworks discover `test_*` methods,
and how a CLI can map `--max-retries 5` onto `job.max_retries` without a giant `if`/`elif`.

In [ ]:
print(getattr(job_b, 'name'))
print(getattr(job_b, 'owner', 'unassigned'))   # default instead of AttributeError

setattr(job_b, 'owner', 'aditya')              # e.g. applying {'owner': 'aditya'} from an API payload
print(vars(job_b))

print(hasattr(job_b, 'owner'))
delattr(job_b, 'owner')
print(hasattr(job_b, 'owner'))

## Methods

A **method** is a function defined inside a class body. It is how behaviour travels with the
data. Python has three kinds — this section covers the first; the other two get their own
sections after encapsulation:

| Kind | First parameter | Bound to | Typical job |
|---|---|---|---|
| **instance method** | `self` (the object) | one instance | read/change *this* object's state |
| **class method** | `cls` (the class) | the class | alternative constructors, class-wide state |
| **static method** | none | nothing | pure helper that belongs with the class |

### Instance methods

The default kind: plain `def` inside the class, `self` first.

In [ ]:
class Job:
    """A unit of background work in a task queue."""

    def __init__(self, name, max_retries=3):
        self.name = name
        self.max_retries = max_retries
        self.attempts = 0

    def record_attempt(self):
        self.attempts += 1                       # methods may change instance state...

    def exhausted(self):
        return self.attempts >= self.max_retries  # ...or just read it

    def describe(self):
        return f'{self.name}: attempt {self.attempts}/{self.max_retries}'


job = Job('resize-image', max_retries=2)
job.record_attempt()
print(job.describe())

print(Job.describe(job))   # the same call spelled through the class - proof of what `self` is

job.record_attempt()
print(job.describe())
print('give up?', job.exhausted())

The `Job` class now has state and behaviour. But watch what happens the moment you try to
*look at* one:

In [ ]:
print(job)      # the default: class name + memory address - tells you nothing useful
print([job])    # and inside a container it is just as opaque

---

## `__repr__` vs `__str__` — define `__repr__` first

That `<__main__.Job object at 0x...>` is the default representation inherited from `object`.
Python has **two** hooks for turning an object into text, and **`__repr__` is the more
important of the two** — the one you should write for every class.

| | `__str__` | `__repr__` |
|---|---|---|
| **Audience** | End users | **Developers** |
| **Goal** | Readable | **Unambiguous** |
| **Called by** | `print()`, `str()`, f-strings | `repr()`, the REPL, debuggers, **containers** |
| **Ideal output** | `"200 from /api/users"` | `HttpResponse(status=200, url='/api/users')` |
| **If missing** | Falls back to `__repr__` | Falls back to `<Thing object at 0x7f...>` |

Three reasons `__repr__` wins:

1. **`__str__` falls back to `__repr__`, not the other way round.** Define `__repr__` and
   you get both. Define only `__str__` and your debugger still shows you a memory address.
2. **Containers use `__repr__`.** Printing a *list* of your objects calls `__repr__` on each
   one — so a list of objects with no `__repr__` is a wall of `<object at 0x...>`.
3. **It is what you see when something goes wrong** — in logs, tracebacks and the debugger.

> **The convention:** `__repr__` should ideally look like the code that would recreate the
> object. Where that isn't practical, use angle brackets for the parts you're summarising:
> `<Connection to db01, 4 active>`.

In [ ]:
class HttpResponse:
    """The kind of object you stare at in a debugger at 2am."""

    def __init__(self, status: int, url: str, body: str) -> None:
        self.status = status
        self.url = url
        self.body = body

    def __repr__(self) -> str:
        # Unambiguous, for developers. Ideally looks like the call that rebuilds it.
        return f"HttpResponse(status={self.status}, url={self.url!r}, body=<{len(self.body)} bytes>)"

    def __str__(self) -> str:
        # Readable, for end users.
        return f"{self.status} from {self.url}"


response = HttpResponse(200, "https://api.example.com/users", "x" * 4096)

print("print(obj)   :", response)              # __str__
print("str(obj)     :", str(response))         # __str__
print("repr(obj)    :", repr(response))        # __repr__
print("f'{obj}'     :", f"{response}")         # __str__
print("f'{obj!r}'   :", f"{response!r}")       # __repr__

# ⚠️ Containers ALWAYS use __repr__ of their elements, never __str__
responses = [response, HttpResponse(404, "https://api.example.com/missing", "")]
print("\ninside a list:")
for r in responses:
    print("  ", r)          # __str__ - because we printed each one
print("\nprinting the list itself:")
print(" ", responses)       # __repr__ for each element

# This is why __repr__ matters more: it is what you see when debugging
print("\nin a dict    :", {"latest": response})


# ---- Without __repr__, this is what you get ----
class Unhelpful:
    def __init__(self, status): self.status = status

print("\nno __repr__  :", [Unhelpful(200), Unhelpful(404)])
print("             ^ two objects, no way to tell them apart")


# ---- The fallback rule ----
class OnlyRepr:
    def __repr__(self): return "OnlyRepr(<useful details>)"

class OnlyStr:
    def __str__(self): return "OnlyStr(<pretty>)"

print("\n__repr__ only, printed:", OnlyRepr(), " <- str() falls back to __repr__")
print("__str__ only, repr'd  :", repr(OnlyStr()), " <- repr() does NOT fall back")

Applying the lesson to our running example — from here on, `Job` always carries a `__repr__`:

In [ ]:
class Job:
    """A unit of background work in a task queue."""

    def __init__(self, name, max_retries=3):
        self.name = name
        self.max_retries = max_retries
        self.attempts = 0

    def __repr__(self):
        return f'Job({self.name!r}, max_retries={self.max_retries})'


queue = [Job('resize-image'), Job('send-invoice', max_retries=5)]
print(queue)          # a readable queue, because containers call __repr__

## Encapsulation

Encapsulation is the idea of wrapping data and the methods that work on it into one unit, and
then **controlling how the data is reached** — so internals can change or validate without
breaking every caller.

### Access modifiers — the honest version

Languages like Java enforce `public`/`protected`/`private` in the compiler. Python does not.

⚠️ Python has **no mechanism that actually enforces access restriction**. It prescribes a
**naming convention** — a leading underscore signals intent — and the interpreter enforces
almost none of it.

| Marker | Called | What Python actually does |
|---|---|---|
| `name` | public | Nothing — accessible everywhere. The default. |
| `_name` | protected | **Nothing.** Pure convention: "internal — for this class and its subclasses". Linters and code review enforce it; the interpreter never will. |
| `__name` | private | **Name mangling:** inside class `A`, `__var` is stored as `_A__var`. Accidental outside access (and subclass name clashes) get harder — but `obj._A__var` still reaches it. |

Python has no truly private instance variables. All three levels, on one realistic class:

In [ ]:
class ApiClient:
    def __init__(self, base_url, token):
        self.base_url = base_url       # public: part of the class's interface
        self._session_id = 'sess-042'  # protected: internal detail, convention only
        self.__token = token           # private: name-mangled to _ApiClient__token


client = ApiClient('https://api.example.com', 'sk-live-1234')

print(client.base_url)          # public - as intended
print(client._session_id)       # works! `_name` is a message to developers, not a lock
# print(client.__token)         # AttributeError - not "denied": no attribute __token EXISTS
print(vars(client))             # ...because it was stored under the mangled name:
print(client._ApiClient__token) # reachable - so mangling is not real privacy either

**NOTE: Neither marker is real enforcement** — a protected member (`_name`) is directly
accessible outside the class, and a private member (`__name`) is still reachable through its
name-mangled form (`obj._ClassName__name`). The underscore says *"internal, may change without
notice — touch at your own risk"*, and the enforcement is social: linters, code review, and
team norms.

### Why bother, then? A validation story

Encapsulation in Python is less about hiding and more about **controlling the write path** so
you can validate. Watch the problem happen first:

In [ ]:
class Job:
    def __init__(self, name, timeout):
        self.name = name
        self.timeout = timeout    # seconds the worker may spend on this job


job = Job('resize-image', 30)

job.timeout = -5      # nothing stops nonsense - and a worker will happily
print(job.timeout)    # kill this job instantly, some time later, far from this line

The classic first fix: rename the attribute to `_timeout` ("internal!") and route access
through **getter and setter methods** that validate:

In [ ]:
class Job:
    def __init__(self, name, timeout):
        self.name = name
        self._timeout = timeout                  # internal storage

    def get_timeout(self):
        return self._timeout

    def set_timeout(self, value):
        if value <= 0:
            raise ValueError(f'timeout must be positive, got {value!r}')
        self._timeout = value


job = Job('resize-image', 30)
print(job.get_timeout())

job.set_timeout(60)                              # valid update: fine
print(job.get_timeout())

try:
    job.set_timeout(-5)                          # invalid update: now CAUGHT at the source
except ValueError as exc:
    print('rejected:', exc)

In [ ]:
# But this "fix" broke every existing caller:
# print(job.timeout)   # AttributeError - the attribute was renamed, not "denied"
job.timeout = 50       # ⚠️ WORSE: runs fine, silently creating a NEW unrelated attribute
print(vars(job))       # _timeout AND timeout now coexist - a silent bug, no error anywhere

#### The update was not backward compatible

Every client that used our class must now rewrite `job.timeout` into `job.get_timeout()` and
`job.timeout = v` into `job.set_timeout(v)`. In Java this is why people write getters/setters
for *everything* from day one, just in case.

Python's answer is better: **`property`** lets attribute *syntax* trigger method *calls* — so
you can start with a plain attribute and add validation later, **without changing the client
code at all**.

### `property()`

- **Syntax:** `property(fget=None, fset=None, fdel=None, doc=None)`
- It bundles a getter, setter and deleter behind a single attribute name.
- With no arguments it returns an empty property; `doc` defaults to the getter's docstring.

In [ ]:
class Job:
    def __init__(self, name, timeout):
        self.name = name
        self._timeout = timeout

    def get_timeout(self):
        return self._timeout

    def set_timeout(self, value):
        if value <= 0:
            raise ValueError(f'timeout must be positive, got {value!r}')
        self._timeout = value

    def del_timeout(self):
        del self._timeout

    # attribute-style access, method-powered:
    timeout = property(get_timeout, set_timeout, del_timeout, 'seconds allowed per run')


job = Job('resize-image', 30)
print(job.timeout)        # attribute syntax -> get_timeout() runs

job.timeout = 60          # attribute syntax -> set_timeout(60) runs
print(job.timeout)

try:
    job.timeout = -5      # the old broken write is now INTERCEPTED
except ValueError as exc:
    print('rejected:', exc)

- Note: in `__init__` we assigned to `self._timeout` directly, so no setter ran during object
  creation. If `__init__` assigned to `self.timeout` instead, that line would automatically call
  the setter — a common trick to reuse the validation at construction time.

- A property object also has `.getter()`, `.setter()` and `.deleter()` methods to attach the
  three functions one at a time:

```python
timeout = property(get_timeout, set_timeout, del_timeout)
```
is equivalent to
```python
timeout = property()
timeout = timeout.getter(get_timeout)
timeout = timeout.setter(set_timeout)
timeout = timeout.deleter(del_timeout)
```

- And *that* shape — "wrap a function, get back a replacement" — is exactly what decorator
  syntax expresses (see **4.4**), which gives us the form used in real code:

### `@property` — the idiomatic spelling

In [ ]:
class Job:
    def __init__(self, name, timeout):
        self.name = name
        self._timeout = timeout

    @property
    def timeout(self):
        """Seconds allowed per run."""
        return self._timeout

    @timeout.setter
    def timeout(self, value):
        if value <= 0:
            raise ValueError(f'timeout must be positive, got {value!r}')
        self._timeout = value

    @timeout.deleter
    def timeout(self):
        del self._timeout


job = Job('resize-image', 30)
print(job.timeout)

job.timeout = 60
print(job.timeout)

try:
    job.timeout = -5
except ValueError as exc:
    print('rejected:', exc)

# A property can also be COMPUTED - no stored attribute at all:
class Retry:
    def __init__(self, attempt):
        self.attempt = attempt

    @property
    def delay(self):                 # read-only: no setter defined
        return 2 ** self.attempt     # derived from other state on demand

r = Retry(attempt=3)
print(r.delay)
try:
    r.delay = 99                     # no setter -> assignment is refused
except AttributeError as exc:
    print('read-only:', exc)

Clients kept writing `job.timeout` the whole time — the class gained validation with **zero
client-side changes**. That is the property payoff, and it cuts both ways:

> **Best practice:** start with plain public attributes. Add `@property` only when you actually
> need validation, a computed value, or a read-only view. You can always add it later without
> breaking anyone — that is the whole point. Getter/setter pairs "just in case" are Java
> reflexes, not Python.

## Class variables — state shared by all instances

An instance variable belongs to one object. A **class variable** is defined in the class body,
outside any method, and is **shared by every instance** — there is exactly one copy, stored on
the class.

**Real uses:** defaults shared by all instances (`default_timeout`), counters
(`instances_created`), registries, and constants that belong with the class
(`MAX_QUEUE_DEPTH = 10_000`).

Lookup rule: `obj.attr` checks the **instance first, then the class**. That one rule explains
everything below.

In [ ]:
class Job:
    max_retries = 3                      # class variable: one copy, on the class

    def __init__(self, name):
        self.name = name                 # instance variable: per object

    def __repr__(self):
        return f'Job({self.name!r})'


a = Job('resize-image')
b = Job('send-invoice')

print(Job.max_retries, a.max_retries, b.max_retries)   # all read the SAME class copy
print(vars(a))          # the instances do not store it at all...
print(Job.__dict__['max_retries'])                     # ...the class does

In [ ]:
# Changing it VIA THE CLASS is seen by every instance:
Job.max_retries = 5
print(a.max_retries, b.max_retries)

# ⚠️ "Changing" it VIA AN INSTANCE does something very different:
a.max_retries = 10       # creates a NEW instance variable on `a` that SHADOWS the class one
print(vars(a))           # now stored on the instance
print(a.max_retries, b.max_retries, Job.max_retries)   # a sees 10; b still follows the class

- Assignment through an instance never modifies the class variable — it creates an instance
  variable of the same name that **shadows** it (instance is checked first). `b` and every
  future instance still follow the class copy; `a` has gone its own way.

⚠️ **The genuinely dangerous version of this trap** is a *mutable* class variable. Mutating
(`.append`) is not assignment — no shadowing instance variable is created, so every instance
mutates the one shared object (the same aliasing trap as **2.7**'s mutable default argument):

In [ ]:
class Deployment:
    tags = []                       # ⚠️ ONE list, shared by every deployment

    def __init__(self, env):
        self.env = env


staging = Deployment('staging')
prod = Deployment('prod')

staging.tags.append('hotfix')       # .append mutates the SHARED list (no assignment -> no shadowing)
print('prod.tags leaked:', prod.tags)


# The fix: mutable per-instance state belongs in __init__
class FixedDeployment:
    def __init__(self, env):
        self.env = env
        self.tags = []              # a fresh list per instance


staging = FixedDeployment('staging')
prod = FixedDeployment('prod')
staging.tags.append('hotfix')
print('prod.tags now  :', prod.tags)

> **Version note (3.14):** annotations on class attributes (`max_retries: int = 3`) are now
> evaluated lazily (PEP 649/749) — `Job.__annotations__` is computed on first access instead of
> at class-creation time. Normal code behaves identically; only tools that introspect
> annotations at import time notice.

## Class methods

A **class method** is bound to the class rather than to an instance. Decorate with
`@classmethod`; the first parameter is the class itself, named `cls` by convention. It can be
called without any instance existing, and it can read or change class-level state.

#### The killer use case: alternative constructors (factory methods)

`__init__` gives a class exactly **one** construction signature. Real data arrives in many
shapes — a CSV row from a queue dump, a decoded JSON payload, a config entry. The Python idiom
is one `from_*` class method per shape (exactly how the stdlib does it:
`datetime.fromtimestamp`, `dict.fromkeys`, `Decimal.from_float`).

In [ ]:
class Job:
    def __init__(self, name, max_retries=3):
        self.name = name
        self.max_retries = max_retries

    def __repr__(self):
        return f'{type(self).__name__}({self.name!r}, max_retries={self.max_retries})'

    @classmethod
    def from_line(cls, line):
        """Alternative constructor: parse a 'name,max_retries' queue-dump row."""
        name, retries = line.split(',')
        return cls(name, int(retries))          # cls, NOT Job - see below

    @classmethod
    def from_dict(cls, payload):
        """Alternative constructor: from a decoded JSON payload."""
        return cls(payload['name'], payload.get('max_retries', 3))


print(Job.from_line('resize-image,5'))                      # no instance needed
print(Job.from_dict({'name': 'send-invoice'}))              # missing key -> default

In [ ]:
# Why `cls` instead of hardcoding `Job`: subclasses inherit the factory CORRECTLY.
class UrgentJob(Job):
    pass

job = UrgentJob.from_line('page-oncall,1')
print(job, '| type:', type(job).__name__)    # an UrgentJob, not a Job - because cls was UrgentJob

> **Version note (3.11+):** the idiomatic return annotation for an alternative constructor is
> `Self` (`from typing import Self`), which makes type checkers understand the
> subclass-returns-subclass behaviour you just saw. Details in **16.4**.

## Static methods

A **static method** receives no automatic first argument at all — no `self`, no `cls`.
Decorate with `@staticmethod`. It is an ordinary function that lives *inside* the class purely
because it **belongs with it conceptually** — a utility that callers should find next to the
class, but that needs nothing from any instance.

In [ ]:
class Job:
    def __init__(self, name, max_retries=3):
        self.name = name
        self.max_retries = max_retries

    @staticmethod
    def backoff_delay(attempt):
        """Seconds to wait before retry N - a pure calculation, no instance needed."""
        return 2 ** attempt


# Callable on the class...
print([Job.backoff_delay(n) for n in range(4)])

# ...and on an instance (same function either way - nothing is passed implicitly):
job = Job('resize-image')
print(job.backoff_delay(3))

### Instance vs class vs static — choosing

| | Needs instance state? | Needs class state? | First param | Typical example |
|---|---|---|---|---|
| **instance method** | yes | (via `type(self)`) | `self` | `job.record_attempt()` |
| **class method** | no | yes | `cls` | `Job.from_line(row)` |
| **static method** | no | no | — | `Job.backoff_delay(3)` |

Rules of thumb:
- Touches `self` → instance method.
- Constructs instances or works with class-wide state → class method.
- Neither, but conceptually belongs with the class → static method. (If it doesn't even belong
  with the class, make it a module-level function.)

## Inheritance

Inheritance models an **is-a** relationship: the subclass *is a* specialised version of the
base class.

- **Base class** (also: parent / super class) — the class being inherited from.
- **Derived class** (also: child / sub class) — the class that inherits, keeps everything the
  base defines, and can **add** members or **override** them.

<img src='./Image/5.1 Image b.jpg' width=18% height=20%/>

**Why it matters in real code:** frameworks hand you a base class and you subclass it —
`unittest.TestCase`, Django's `Model`, `http.server.BaseHTTPRequestHandler`. Exceptions are a
class hierarchy (**06**). And your own code uses it whenever several variants share a core:
one notification system, many channels; one storage API, many backends.

### The shapes of inheritance

<img src='./Image/5.1 Image a.jpg' width=75% height=50%/>

We will build each shape out of one domain: **notification channels** — the part of a system
that tells humans their build failed.

### Single inheritance

One base, one derived: `class Derived(Base)`. The subclass gets every attribute and method of
the base for free, and adds its own:

In [ ]:
class Notifier:
    """Base: any channel that can deliver a message."""

    def __init__(self, service):
        self.service = service
        self.sent = 0

    def send(self, message):
        self.sent += 1
        print(f'[{self.service}] {message}')


class EmailNotifier(Notifier):            # EmailNotifier IS A Notifier
    def __init__(self, recipient):
        Notifier.__init__(self, 'email')  # run the base initialiser explicitly...
        self.recipient = recipient        # ...then add subclass-specific state

    def send(self, message):              # OVERRIDE: same name, specialised behaviour
        Notifier.send(self, f'to={self.recipient} subject="Build update" | {message}')


email = EmailNotifier('dev-team@example.com')
email.send('build #4812 failed on main')
email.send('build #4813 green')
print(email.sent, 'messages sent')        # `sent` came from the base class

### Method overriding and `super()`

`EmailNotifier` defines `__init__` and `send` even though `Notifier` already has them. When
both classes define a method, the derived class's version **overrides** the base's — lookup
finds the subclass version first.

Generally an override should **extend** the base behaviour, not secretly replace it — which
means calling the base version from inside the override. Above we did that by naming the base
explicitly (`Notifier.__init__(self, ...)`). That works, but hardcodes the parent: rename or
re-parent the class and every call site breaks — and under multiple inheritance it is actually
wrong (it can skip classes or run one twice).

**`super()`** fixes this: it returns a proxy that delegates to *the next class in method
resolution order* — no parent named, no `self` passed:

In [ ]:
class EmailNotifier(Notifier):
    def __init__(self, recipient):
        super().__init__('email')         # no class name, no self
        self.recipient = recipient

    def send(self, message):
        super().send(f'to={self.recipient} subject="Build update" | {message}')


email = EmailNotifier('dev-team@example.com')
email.send('build #4814 flaky, retrying')

### `isinstance()` and `issubclass()`

- **`isinstance(obj, cls)`** — is `obj` an instance of `cls` *or of any class derived from it*?
  This is the practical meaning of is-a: an `EmailNotifier` **is a** `Notifier`, so code written
  against `Notifier` accepts it.
- **`issubclass(sub, sup)`** — is the first class derived from the second?
- Since every class inherits from `object`, everything is an instance of `object`.

In [ ]:
plain = Notifier('console')
email = EmailNotifier('dev-team@example.com')

print(isinstance(email, EmailNotifier))
print(isinstance(email, Notifier))       # subclass instances count: is-a
print(isinstance(plain, EmailNotifier))  # ...but not the other way round

print(issubclass(EmailNotifier, Notifier))
print(issubclass(Notifier, EmailNotifier))
print(issubclass(EmailNotifier, object))  # everything descends from object

### Encapsulation meets inheritance

The underscore conventions from earlier interact with inheritance in two very different ways:

- `_protected` members are **meant** for subclasses — "internal to this class *and its
  children*". They flow down normally.
- `__private` members do **not** flow down usefully, because name mangling bakes the defining
  class's name into the attribute.

In [ ]:
# _protected members: the base's internals are available to the subclass - by design.
class HttpClient:
    def __init__(self, base_url):
        self._base_url = base_url                  # internal, but subclass-visible

    def _headers(self):                            # internal helper
        return {'User-Agent': 'learn-python/5.1'}

    def get(self, path):
        return f'GET {self._base_url}{path} {self._headers()}'


class AuthenticatedClient(HttpClient):
    def __init__(self, base_url, token):
        super().__init__(base_url)
        self._token = token

    def _headers(self):                            # extend the internal helper
        return super()._headers() | {'Authorization': f'Bearer {self._token}'}


api = AuthenticatedClient('https://api.example.com', 'sk-1234')
print(api.get('/health'))
# And, as always, the convention does not stop outsiders:
print(api._base_url)     # works - `_name` is a convention, not enforcement

In [ ]:
# __private members: name mangling makes them effectively invisible to subclasses.
class PaymentGateway:
    def __init__(self):
        self.__api_key = 'sk-live-1234'    # stored as _PaymentGateway__api_key

    def charge(self, amount):
        return f'charged {amount} (signed with {self.__api_key})'   # resolves fine inside


class TestGateway(PaymentGateway):
    def debug_key(self):
        return self.__api_key    # ⚠️ HERE this compiles to self._TestGateway__api_key


gw = TestGateway()
print(gw.charge(99))             # inherited method works: it uses the PARENT's mangled name
# print(gw.debug_key())          # AttributeError: 'TestGateway' object has no attribute
                                 #                 '_TestGateway__api_key' - never set!
print(gw._PaymentGateway__api_key)   # the data exists, under the parent's mangled name

That is the actual purpose of `__name`: not secrecy, but making sure a subclass that happens
to reuse the name cannot *accidentally* clobber the parent's attribute. When you genuinely want
subclasses to share an internal, use a single underscore.

### Hierarchical inheritance

One base, **several sibling** subclasses — the most common shape in practice. The base defines
the shared contract; each sibling specialises it:

In [ ]:
class SlackNotifier(Notifier):
    def __init__(self, channel):
        super().__init__('slack')
        self.channel = channel

    def send(self, message):
        super().send(f'{self.channel} :rotating_light: {message}')


class SmsNotifier(Notifier):
    def __init__(self, number):
        super().__init__('sms')
        self.number = number

    def send(self, message):
        super().send(f'{self.number} {message[:40]}')      # SMS: keep it short


# Code written against the BASE type works with every sibling:
incident = 'db01 disk 95% full - paging on-call'
for notifier in [EmailNotifier('ops@example.com'), SlackNotifier('#alerts'), SmsNotifier('+91-98xxx')]:
    notifier.send(incident)

### Multilevel inheritance

Deriving from a class that is itself derived — a chain, of any depth:

```python
class Base:              pass
class Derived1(Base):     pass
class Derived2(Derived1): pass     # inherits from Base AND Derived1
```

A digest email notifier *is an* email notifier *is a* notifier — it batches messages and sends
one combined email:

In [ ]:
class DigestEmailNotifier(EmailNotifier):        # Notifier -> EmailNotifier -> this
    def __init__(self, recipient, batch_size=3):
        super().__init__(recipient)
        self.batch_size = batch_size
        self.pending = []

    def send(self, message):
        self.pending.append(message)
        if len(self.pending) >= self.batch_size:
            super().send(' // '.join(self.pending))   # one combined email via the parent
            self.pending.clear()


digest = DigestEmailNotifier('ops@example.com', batch_size=3)
digest.send('build #1 failed')       # buffered - nothing printed yet
digest.send('build #2 failed')       # buffered
digest.send('build #3 green')        # third message triggers the combined send
print(isinstance(digest, Notifier), '- still a Notifier, two levels up')

#### Overriding along the chain

When every level overrides the same method and each calls `super()`, the calls run from the
most-derived class **upward**, each layer adding its contribution. This is exactly how logging
handlers, middleware stacks and template-method frameworks are built:

In [ ]:
class LogHandler:
    def emit(self, record):
        print(f'WRITE {record}')


class TimestampedHandler(LogHandler):
    def emit(self, record):
        super().emit(f'2026-08-21T10:00:00 {record}')     # add a timestamp, delegate up


class JsonHandler(TimestampedHandler):
    def emit(self, record):
        super().emit(f'{{"msg": "{record}"}}')            # wrap as JSON, delegate up


JsonHandler().emit('disk full')
# JsonHandler ran first, then TimestampedHandler, then LogHandler - most-derived outward.
# `super().emit(...)` is the modern spelling of super(JsonHandler, self).emit(...) -
# the two-argument form still works and means the same thing.

⚠️ One overriding subtlety: a `__private` method is **not** protected from overriding by some
rule — there is no rule, only name mangling. `__method` in `Parent` is stored as
`_Parent__method`; defining `__method` in the child creates `_Child__method`, a *different*
name, so the parent's own internal calls keep resolving to the parent's version. It merely
*looks* un-overridable. (The underscore appendix at the end shows this trick used deliberately.)

### Multiple inheritance

A class can list **several** bases and inherit from all of them:

```python
class Base1:                  pass
class Base2:                  pass
class Derived(Base1, Base2):  pass
```

Real code uses this mostly for *mixins* — small classes contributing one orthogonal capability
each (**5.2** builds those properly). The mechanics first, with two independent capabilities:

In [ ]:
class Identifiable:
    def __init__(self, uid):
        self.uid = uid

    def ref(self):
        return f'#{self.uid}'


class Timestamped:
    def __init__(self, created_at):
        self.created_at = created_at

    def age_label(self):
        return f'created {self.created_at}'


class Ticket(Identifiable, Timestamped):       # inherits from BOTH
    def __init__(self, uid, created_at, title):
        Identifiable.__init__(self, uid)       # with disjoint bases, explicit calls
        Timestamped.__init__(self, created_at) # to each parent are the simple option
        self.title = title


t = Ticket(101, '2026-08-20', 'Login page returns 500')
print(t.ref(), '|', t.age_label(), '|', t.title)

That worked because the two bases touch **different** attributes. Watch what happens when the
bases overlap:

In [ ]:
class DefaultConfig:
    def __init__(self):
        self.source = 'defaults'
        self.timeout = 30


class EnvConfig:
    def __init__(self):
        self.source = 'env'
        self.timeout = 60


class AppConfig(DefaultConfig, EnvConfig):
    def __init__(self):
        DefaultConfig.__init__(self)
        EnvConfig.__init__(self)       # ⚠️ runs second, so ITS values win


cfg = AppConfig()
print(cfg.source, cfg.timeout)   # 'env' 60 - decided by the ORDER OF THE CALLS,
                                 # not by the order of the bases in `class AppConfig(...)`

Both `__init__`s assigned `self.source` — last writer wins. With explicit parent calls,
*you* are hand-managing the initialisation order, and nothing checks you got it right (or that
you didn't run a shared grandparent twice). This is the problem the **MRO** and `super()`
solve properly.

### Method Resolution Order (MRO)

- MRO is the order in which Python looks up a method or attribute in a hierarchy of classes.
    - This order is also called the **linearization** of the class, and the algorithm that
      computes it is **C3 linearization**.
- ⚠️ Python 3 does **not** use plain depth-first, left-to-right search — that was the lookup
  rule of Python 2's old-style classes. C3 starts out looking depth-first and left-to-right,
  but it guarantees more:
    - a class is always checked **before** its parents,
    - the left-to-right order of the bases in the `class` statement is preserved, and
    - each class appears **exactly once** in the order.
- The two rules diverge exactly in **diamond** hierarchies: depth-first would visit a shared
  grandparent before an unvisited parent, while C3 defers the shared base until every class
  that inherits from it has been checked. (Notebook **5.2** walks through a real diamond step
  by step.)
- Inspect the order any time with `ClassName.__mro__` or `ClassName.mro()`.
- For `AppConfig(DefaultConfig, EnvConfig)`, the order is: AppConfig, DefaultConfig,
  EnvConfig, object.

In [ ]:
print(AppConfig.__mro__)

In [ ]:
# super() follows the MRO - one step at a time, each class exactly once.
class AppConfig(DefaultConfig, EnvConfig):
    def __init__(self):
        super().__init__()     # delegates to the NEXT class in the MRO: DefaultConfig


cfg = AppConfig()
print(cfg.source, cfg.timeout)   # 'defaults' 30 - only DefaultConfig.__init__ ran,
                                 # because DefaultConfig.__init__ never calls super() itself

print(AppConfig.__mro__)

Only `DefaultConfig.__init__` ran: `super()` handed control to the next class in the MRO, and
that class — written without a `super().__init__()` call of its own — ended the chain. For the
whole chain to run, **every class in a cooperative hierarchy must call `super().__init__()`**,
including the bases. That discipline, and when it is worth it, is a core topic of **5.2**.

> The MRO is not trivia — it is the answer to "which method actually runs?" in any
> non-trivial hierarchy, and the first thing to print when multiple inheritance surprises you.

## Composition — *part of*

Inheritance is not the only way to reuse behaviour, and it is frequently the wrong one.
**Composition** models a **part-of** relationship: a class contains an object of another class
and delegates work to it.

- The containing class is the **composite**; the contained class is the **component**.
- The composite *owns* its component: it creates it internally, and when the composite dies,
  the component dies with it.

<img src='./Image/5.1 Image c.jpg' width=20% height=20%/>

- The `1` in the diagram is the **cardinality**: how many components the composite holds
  (a number, `*` for any number, or a range like `1..4`).

An HTTP client *has a* retry policy. A retry policy is not a client, and a client is not a
policy — no is-a anywhere, so inheritance would be a lie. Composition says it straight:

In [ ]:
class RetryPolicy:
    """How a client retries: the component."""

    def __init__(self, max_attempts=3, base_delay=1.0):
        self.max_attempts = max_attempts
        self.base_delay = base_delay

    def delays(self):
        return [self.base_delay * 2 ** n for n in range(self.max_attempts)]


class HttpClient:
    """The composite: creates and owns its RetryPolicy."""

    def __init__(self, base_url, max_attempts=3):
        self.base_url = base_url
        self.retry = RetryPolicy(max_attempts)      # component built INSIDE the composite

    def get(self, path):
        # delegate the retry maths to the component:
        return f'GET {self.base_url}{path} (retry delays: {self.retry.delays()}s)'


client = HttpClient('https://api.example.com', max_attempts=4)
print(client.get('/users'))

Points to notice:
- No `RetryPolicy` object exists outside the client — it was instantiated inside `__init__`.
  When the client is garbage-collected, its policy goes with it.
- Swapping retry behaviour means passing different *numbers* (or a different policy object) —
  no subclass explosion. This "inject the varying part" idea scales into the
  **composition-over-inheritance** principle that **5.2** and **5.4** develop fully.

## Aggregation — *has a*

Aggregation is the looser cousin: the container **uses** an object it did **not** create and
does not own. The associated objects have independent lifetimes — delete the container and the
component lives on (it usually arrived as a constructor argument and is often **shared**):

In [ ]:
class ConnectionPool:
    """An expensive shared resource, created once at application startup."""

    def __init__(self, size):
        self.size = size

    def __repr__(self):
        return f'ConnectionPool(size={self.size})'


class UserService:
    def __init__(self, pool):
        self.pool = pool            # RECEIVED, not created: aggregation


class ReportService:
    def __init__(self, pool):
        self.pool = pool            # the same pool, shared


pool = ConnectionPool(10)
users = UserService(pool)
reports = ReportService(pool)

print(users.pool is reports.pool)   # one pool, two aggregating services

del users                           # the service dies...
print(pool)                         # ...the pool survives independently

| | Composition | Aggregation |
|---|---|---|
| Relationship | part-of | has-a |
| Who creates the component | the composite, internally | someone else; passed in |
| Lifetimes | tied — component dies with composite | independent |
| Sharing | component belongs to one composite | component freely shared |

Both beat inheritance whenever the honest sentence is "*X has a Y*" rather than "*X is a Y*".

## Abstraction and abstract base classes

**Abstraction** means exposing *what* an object does while hiding *how* — callers see
`storage.save(key, data)` and never care whether bytes land on disk, in S3, or in a dict.
Encapsulation is the mechanism; abstraction is the design payoff.

To make abstraction real you need a **contract**: "every storage backend MUST provide `save`
and `load`". A plain base class cannot enforce that:

In [ ]:
class StorageBackend:
    """A wishful contract: subclasses are SUPPOSED to implement these."""

    def save(self, key, data):
        pass

    def load(self, key):
        pass


class DiskStorage(StorageBackend):
    def __init__(self, root):
        self.root = root
    # ...forgot to implement save() and load()!


# Nothing enforces anything:
useless = StorageBackend()        # the "template" itself can be instantiated
broken = DiskStorage('/var/data')
print(broken.load('user:42'))     # silently returns None - the bug surfaces far away

Two silent failures: the template is instantiable, and a half-finished subclass slips through,
returning `None` into code that expected data.

### The `abc` module

Python ships the fix: the **module** is `abc` (lowercase), providing the base **class** `ABC`
and the `@abstractmethod` decorator.

- Inherit from `ABC` and mark the contract methods `@abstractmethod`.
- A class with unimplemented abstract methods **cannot be instantiated** — attempting it raises
  `TypeError` *at construction time*, naming exactly what is missing.
- Abstract classes may still carry real implementations — shared helpers and defaults that
  subclasses inherit. That makes ABCs a good home for a common API that third parties
  implement, such as a plugin interface.

In [ ]:
from abc import ABC, abstractmethod


class StorageBackend(ABC):
    @abstractmethod
    def save(self, key, data): ...

    @abstractmethod
    def load(self, key): ...

    def refresh(self, key, data):          # concrete method: shared by ALL backends
        self.save(key, data)
        return self.load(key)


class DiskStorage(StorageBackend):
    def __init__(self, root):
        self.root = root
    # still forgot save() and load()...


try:
    StorageBackend()                       # the template can no longer be instantiated
except TypeError as exc:
    print('template :', exc)

try:
    DiskStorage('/var/data')               # the half-finished subclass fails LOUDLY, at
except TypeError as exc:                   # construction - not None-ing around at runtime
    print('subclass :', exc)

In [ ]:
class InMemoryStorage(StorageBackend):
    """A complete implementation - handy for tests."""

    def __init__(self):
        self._data = {}

    def save(self, key, data):
        self._data[key] = data

    def load(self, key):
        return self._data[key]


store = InMemoryStorage()               # all abstract methods implemented: instantiable
store.save('user:42', {'name': 'aditya'})
print(store.load('user:42'))
print(store.refresh('user:42', {'name': 'priya'}))   # inherited concrete helper works too

## Polymorphism: poly = many, morph = forms

Polymorphism is the ability of **one operation to work across many types** — the same call,
resolved to different behaviour depending on what it lands on. You have been using it all
along: `len('deploy')`, `len([1, 2])` and `len({'env': 'prod'})` are one function name over
three types.

Python gives you it in three flavours:

1. **Method overriding** (just covered): code written against `Notifier` ran `send()` on email,
   Slack and SMS objects without knowing which it had.
2. **Duck typing** (no inheritance needed): if two unrelated classes both provide `render()`,
   any code that calls `render()` accepts both. "If it quacks like a duck…" — the types never
   have to declare a common base. This is Python's native style, formalised by protocols in **5.4**.
3. **Operator overloading** (next section): `+` means addition for `int`, concatenation for
   `str`, merging for `list` — and whatever you decide for your own types.

⚠️ **Method *overloading*** — several methods with the same name but different parameter lists,
as in C++/Java — does **not** exist in Python: a later `def` of the same name simply replaces
the earlier one. Defaults, `*args`, and `@functools.singledispatch` cover those use cases.

> **Related, not identical:** *generic programming* - one class parameterised over the element
> type it works with, like `list[int]` vs `list[str]`. Python 3.12's PEP 695 lets you write
> your own (`class Stack[T]:`) - that is notebook **16.3**'s territory.

In [ ]:
# Duck typing: unrelated classes, one shared method name - no common base class.
import json


class TextReport:
    def render(self, stats):
        return f"jobs={stats['jobs']} failed={stats['failed']}"


class JsonReport:
    def render(self, stats):
        return json.dumps(stats)


stats = {'jobs': 120, 'failed': 3}
for report in [TextReport(), JsonReport()]:
    print(report.render(stats))        # the caller neither knows nor cares which class

### Operator Overloading

Every operator in Python is a method call in disguise. Defining that method for your class is
**operator overloading**: your objects gain `+`, `<`, `==`, `in` … with meanings you choose.

The stdlib leans on this everywhere — you have already used all of these:

```python
datetime.now() + timedelta(hours=2)     # __add__ on dates
Path('/var') / 'log' / 'app.log'        # __truediv__ builds paths
b'GET' + b' /health'                    # __add__ concatenates
{'a': 1} | {'b': 2}                     # __or__ merges dicts (3.9+)
```

To watch the mechanics, we use the classic **2-D grid point** — classic for a reason: the
arithmetic is so obvious that the *machinery* stays visible. (A real-world type follows right
after.)

In [ ]:
class GridPoint:
    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'GridPoint({self.x}, {self.y})'

In [ ]:
pt1 = GridPoint(3, 5)
pt2 = GridPoint(-1, 4)
pt3 = pt1 + pt2      # you and I know what this SHOULD mean...
print(pt3)

`TypeError: unsupported operand type(s) for +` — Python has no idea how to add two
`GridPoint` objects, and refuses to guess.

#### Special (dunder) methods are the hook

Class methods that begin and end with double underscores are **special functions**: hooks that
Python calls on your behalf when built-in syntax touches your object. You already know
`__init__` and `__repr__`. The one behind `+` is `__add__`:

```
pt1 + pt2      is evaluated as      pt1.__add__(pt2)
 ─┬─  ─┬─                            ─┬─         ─┬─
  │    └─ becomes the `other` argument            │
  └─ becomes `self` ──────────────────────────────┘
```

Define `__add__`, and `+` starts working. Two conventions to keep it well-behaved:
- **Return a new object** rather than mutating `self` — `+` is expected to be side-effect free.
- For operands you don't understand, return **`NotImplemented`** (a special constant, not an
  exception) so Python can try the other operand's reflected method before giving up.

In [ ]:
class GridPoint:
    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'GridPoint({self.x}, {self.y})'

    def __add__(self, other):                    # the hook behind `+`
        if not isinstance(other, GridPoint):
            return NotImplemented                # let Python try other.__radd__ / fail cleanly
        return GridPoint(self.x + other.x, self.y + other.y)   # NEW object, no mutation


pt1 = GridPoint(3, 5)
pt2 = GridPoint(-1, 4)
print(pt1 + pt2)                  # sugar for pt1.__add__(pt2)
print(pt1.__add__(pt2))           # the same call, spelled out

#### The same idea on a real type

A `Duration` — a length of time, as used for timeouts and retry delays. Adding durations is
meaningful; adding a duration to a dict is not. And a container of them should `sum()`:

In [ ]:
class Duration:
    """A length of time, e.g. a retry delay or a timeout budget."""

    def __init__(self, seconds):
        self.seconds = seconds

    def __repr__(self):
        return f'Duration({self.seconds}s)'

    def __add__(self, other):
        if not isinstance(other, Duration):
            return NotImplemented
        return Duration(self.seconds + other.seconds)


print(Duration(30) + Duration(90))          # a timeout budget from two parts

# ⚠️ but sum() starts from 0, and 0 + Duration(1) is NOT our __add__'s job:
delays = [Duration(1), Duration(2), Duration(4)]
try:
    print(sum(delays))
except TypeError as exc:
    print('sum() failed:', exc)

`sum()` computes `0 + Duration(1) + ...`. For `0 + Duration(1)`, Python first asks
`int.__add__(0, d)` — which returns `NotImplemented` — and then asks the **reflected** method
`Duration.__radd__(d, 0)`. We never wrote one, so the whole expression fails.

Every arithmetic dunder has an `r`-twin (`__radd__`, `__rsub__`, …) that runs when your object
sits on the **right** side and the left side gave up. Supporting `sum()` takes three lines:

In [ ]:
class Duration:
    def __init__(self, seconds):
        self.seconds = seconds

    def __repr__(self):
        return f'Duration({self.seconds}s)'

    def __add__(self, other):
        if not isinstance(other, Duration):
            return NotImplemented
        return Duration(self.seconds + other.seconds)

    def __radd__(self, other):
        if other == 0:                  # the start value sum() uses
            return self
        return NotImplemented


delays = [Duration(1), Duration(2), Duration(4)]
print(sum(delays))                      # 0 + 1s + 2s + 4s

#### The mathematical operators and their hooks

| Operator | Operation | Method |
|---|---|---|
| `+` | addition | `__add__(self, other)` |
| `-` | subtraction | `__sub__(self, other)` |
| `*` | multiplication | `__mul__(self, other)` |
| `/` | true division | `__truediv__(self, other)` |
| `//` | floor division | `__floordiv__(self, other)` |
| `%` | remainder | `__mod__(self, other)` |
| `**` | power | `__pow__(self, other)` |
| `&` / `\|` / `^` | bitwise and/or/xor | `__and__` / `__or__` / `__xor__` |

Each also has a reflected `__r*__` form (right-hand operand) and an in-place `__i*__` form
(`+=` → `__iadd__`; without it, `a += b` falls back to `a = a + b`).

#### Relational operators

Comparisons overload the same way — the methods usually return `True`/`False` rather than a
new instance:

| Operator | Operation | Method |
|---|---|---|
| `>` | greater than | `__gt__(self, other)` |
| `>=` | greater or equal | `__ge__(self, other)` |
| `<` | less than | `__lt__(self, other)` |
| `<=` | less or equal | `__le__(self, other)` |
| `==` | equal | `__eq__(self, other)` |
| `!=` | not equal | `__ne__(self, other)` |

In [ ]:
class GridPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'GridPoint({self.x}, {self.y})'

    def __gt__(self, other):        # compare by distance from the origin
        if not isinstance(other, GridPoint):
            return NotImplemented
        return self.x**2 + self.y**2 > other.x**2 + other.y**2


pt1 = GridPoint(3, 5)
pt2 = GridPoint(-1, 4)
print(pt1 > pt2)      # sugar for pt1.__gt__(pt2)
print(pt1 < pt2)      # no __lt__ defined - Python swaps the operands and uses __gt__

The second line worked with **no `__lt__` at all**: for `pt1 < pt2` Python is happy to
evaluate the reflected form `pt2 > pt1` instead. Equality, though, has a trap severe enough to
deserve its own section.

---

## 🔴 `__eq__` and `__hash__` must be defined together

This is one of the most damaging silent traps in Python OOP.

**When you define `__eq__`, Python automatically sets `__hash__ = None`.**

Your class instantly becomes **unhashable** — it can no longer be a dict key or a set
member — and nothing warns you until something fails at run time, often far away.

The reason is a rule the language must uphold:

> If `a == b`, then `hash(a) == hash(b)`.

By default, objects hash by identity and compare by identity, so the rule holds. The moment
you redefine equality, the default hash is inconsistent with it — so Python removes it
rather than let you build a broken dict.

| You define | Python does | Result |
|---|---|---|
| Nothing | identity `__eq__` + identity `__hash__` | Hashable |
| `__eq__` only | sets `__hash__ = None` | ⚠️ **Unhashable** |
| `__eq__` and `__hash__` | uses yours | Hashable, correctly |
| `@dataclass(frozen=True)` | generates both | Hashable (see **5.3**) |

Two rules that follow:
1. Hash over **exactly the fields** you compare in `__eq__`.
2. Only hash things that **don't change** — mutating a hashed field loses the object inside
   the dict.

In [ ]:
# A cache key, of the kind you would use in a request-level cache.
class CacheKey:
    def __init__(self, endpoint: str, user_id: int) -> None:
        self.endpoint = endpoint
        self.user_id = user_id

    def __repr__(self) -> str:
        return f"CacheKey({self.endpoint!r}, {self.user_id})"

    def __eq__(self, other) -> bool:
        if not isinstance(other, CacheKey):
            return NotImplemented
        return (self.endpoint, self.user_id) == (other.endpoint, other.user_id)


a = CacheKey("/api/profile", 42)
b = CacheKey("/api/profile", 42)

print("a == b :", a == b, " <- __eq__ works")
print("a is b :", a is b)

# ...but the moment you try to USE it as a key:
try:
    cache = {a: "cached response"}
except TypeError as exc:
    print("\nusing it as a dict key:", exc)

print("__hash__ is now:", CacheKey.__hash__, " <- Python set it to None for us")


# ---- The fix: define __hash__ over the SAME fields as __eq__ ----
class FixedCacheKey:
    def __init__(self, endpoint: str, user_id: int) -> None:
        self.endpoint = endpoint
        self.user_id = user_id

    def _key(self) -> tuple[str, int]:
        return (self.endpoint, self.user_id)

    def __repr__(self) -> str:
        return f"CacheKey({self.endpoint!r}, {self.user_id})"

    def __eq__(self, other) -> bool:
        if not isinstance(other, FixedCacheKey):
            return NotImplemented
        return self._key() == other._key()

    def __hash__(self) -> int:
        return hash(self._key())


a = FixedCacheKey("/api/profile", 42)
b = FixedCacheKey("/api/profile", 42)

cache: dict[FixedCacheKey, str] = {a: "cached response"}
print("\nlookup with an EQUAL but distinct object:", cache[b])
print("dedup in a set                          :", len({a, b}))

# ⚠️ If you mutate a field that feeds the hash, the object is lost inside the dict
a.user_id = 99
print("\nafter mutating a.user_id:")
try:
    cache[a]
except KeyError:
    print("  the key is now unfindable - hash changed while it sat in the dict")
print("  this is exactly why hashable objects should be IMMUTABLE")

### `functools.total_ordering`

Writing all six comparison methods by hand is repetitive and easy to get inconsistent.
`@total_ordering` derives the rest from **`__eq__` plus one** of `__lt__`, `__le__`,
`__gt__` or `__ge__`.

**Real-world use case:** version numbers, priority-queue entries, date ranges, log levels —
anything you need to sort or compare.

In [ ]:
from functools import total_ordering

# Defining __eq__ and __lt__ only, and letting total_ordering fill in the rest.
@total_ordering
class Version:
    """A semantic version, comparable and sortable."""

    def __init__(self, major: int, minor: int, patch: int) -> None:
        self.major, self.minor, self.patch = major, minor, patch

    def _key(self) -> tuple[int, int, int]:
        return (self.major, self.minor, self.patch)

    @classmethod
    def parse(cls, text: str) -> "Version":
        return cls(*(int(part) for part in text.split(".")))

    def __repr__(self) -> str:
        return f"Version({self.major}.{self.minor}.{self.patch})"

    def __eq__(self, other) -> bool:
        if not isinstance(other, Version):
            return NotImplemented
        return self._key() == other._key()

    def __lt__(self, other) -> bool:
        if not isinstance(other, Version):
            return NotImplemented
        return self._key() < other._key()

    def __hash__(self) -> int:
        return hash(self._key())


releases = [Version.parse(v) for v in ["1.2.0", "1.10.0", "1.2.10", "0.9.9", "1.2.0"]]

print("unsorted:", releases)
print("sorted  :", sorted(releases))
print("latest  :", max(releases))

# total_ordering derived <=, >, >= from __lt__ and __eq__
v1, v2 = Version.parse("1.2.0"), Version.parse("1.10.0")
print(f"\n{v1} <  {v2} -> {v1 < v2}")
print(f"{v1} <= {v2} -> {v1 <= v2}   <- derived")
print(f"{v1} >  {v2} -> {v1 > v2}   <- derived")
print(f"{v1} >= {v2} -> {v1 >= v2}   <- derived")

# Because __hash__ is consistent with __eq__, dedup works
print("\nunique releases:", sorted(set(releases)))

# Returning NotImplemented (not raising) lets Python try the reflected operation
print("\nVersion == 'string':", Version.parse("1.0.0") == "1.0.0", " <- falls back to False")

---

## Building your own iterator

`for` loops, unpacking and comprehensions all speak one protocol, and any class can join in by
implementing two methods:

- **`__iter__()`** — called once at the start of iteration; returns the iterator object
  (usually `self`) and is the place to reset any position state.
- **`__next__()`** — called for each step; returns the next value, and **raises
  `StopIteration`** when there is nothing left. The `for` loop catches that exception — that
  is how loops end.

A real place to want one: a **retry schedule**. "Attempt, wait 1s, attempt, wait 2s, wait 4s…
give up after N tries" is exactly a short sequence a `for` loop should drive:

In [ ]:
class RetrySchedule:
    """Iterator of exponential-backoff delays: base, base*2, base*4, ..."""

    def __init__(self, max_attempts, base=1.0):
        self.max_attempts = max_attempts
        self.base = base

    def __iter__(self):
        self.attempt = 0                  # reset position at the start of each loop
        return self

    def __next__(self):
        if self.attempt >= self.max_attempts:
            raise StopIteration           # signals: the for loop ends here
        delay = self.base * 2 ** self.attempt
        self.attempt += 1
        return delay


for delay in RetrySchedule(4):
    print(f'request failed - sleeping {delay:.0f}s before retrying')

In [ ]:
# What the for loop does under the hood:
schedule = iter(RetrySchedule(2))     # iter() calls __iter__
print(next(schedule))                 # next() calls __next__
print(next(schedule))
try:
    next(schedule)                    # exhausted
except StopIteration:
    print('StopIteration - the for loop would end here')

For sequences like this, a **generator function** (`yield`, notebook **4.3**) produces the
same iterator in a third of the code — reach for a class-based iterator when the iteration
state is genuinely complex or the object has other jobs besides iterating. The protocol above
is what both compile down to, and it is what the `Repository` example implements at the end of
this notebook.

---

### `__new__` vs `__init__`

Two steps hide behind `Thing()`:

| Method | Job | Returns |
|---|---|---|
| `__new__(cls, ...)` | **Creates** and returns the new object | the instance |
| `__init__(self, ...)` | **Initialises** the object it was given | `None` |

You have only ever needed `__init__`, and that is normal. `__new__` matters in exactly two
practical situations: **singletons**, and **subclassing an immutable type** (`int`, `str`,
`tuple`) — where by the time `__init__` runs, the value is already fixed.

In [ ]:
# ---- __new__ runs BEFORE __init__, and returns the instance ----
class ConnectionPool:
    """Only one pool should ever exist - a classic singleton."""
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            print("  creating the one and only pool")
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, size: int = 5) -> None:
        # ⚠️ __init__ runs on EVERY call, even when __new__ returned an existing object
        if not hasattr(self, "size"):
            self.size = size

    def __repr__(self) -> str:
        return f"ConnectionPool(size={self.size})"


a = ConnectionPool(10)
b = ConnectionPool(99)

print("\na is b :", a is b)
print("a      :", a)
print("b      :", b, " <- size stayed 10; the second __init__ was guarded")


# ---- __new__ is also how you subclass an immutable type ----
class Port(int):
    """An int that refuses to be an invalid TCP port."""

    def __new__(cls, value):
        value = int(value)
        if not 0 < value < 65536:
            raise ValueError(f"invalid port: {value}")
        return super().__new__(cls, value)      # must happen in __new__, not __init__

    def __repr__(self) -> str:
        return f"Port({int(self)})"


p = Port("8080")
print("\n", p, "| is an int:", isinstance(p, int), "| +1 =", p + 1)

try:
    Port(99999)
except ValueError as exc:
    print("rejected:", exc)

print("""
Rule of thumb: you almost never need __new__.
Reach for it only for singletons, immutable subclasses (int/str/tuple),
and metaclass-adjacent tricks. Everything else belongs in __init__.
""")

### `__del__` — the end of the lifecycle

The counterpart bookend: **`__del__`** runs when an object is *about to be destroyed* — i.e.
when its reference count reaches zero — `del x` removes one *reference*; the object
dies (and `__del__` runs) only when the last reference is gone.

Python's garbage collector handles memory for you, so unlike C++ you will rarely write one.

⚠️ **Do not rely on `__del__` for cleanup that matters** (closing files, connections, locks).
Its timing is an implementation detail — it may run late, at interpreter shutdown, or (for
objects in reference cycles) essentially whenever. Real cleanup belongs in a context manager
(`with`, **06 Exception Handling**), which runs deterministically.

In [ ]:
class Connection:
    def __init__(self, host):
        self.host = host
        print(f'connected to {self.host}')

    def __del__(self):                       # destructor
        print(f'connection to {self.host} closed')


conn = Connection('db01')
backup = conn        # a second reference to the SAME object

del conn             # removes one reference - object lives on, no __del__ yet
print('conn deleted, but backup still holds the object:', backup.host)

del backup           # last reference gone -> refcount hits zero -> __del__ runs

---

## `__slots__` — trading flexibility for memory

By default every instance carries a `__dict__` so you can attach attributes at will. That
flexibility costs memory — irrelevant for ten objects, significant for ten million.

`__slots__` declares the attribute names up front. Python then stores them in a fixed
array instead of a dict.

**Real-world use case:** parsing a multi-gigabyte log file into record objects, holding
market ticks, or any ingest pipeline creating millions of small instances. It is a
profiling-driven optimisation, not a default.

A useful side effect: **typos become errors** rather than silently creating a new attribute.

In [ ]:
import sys

# ---- Without __slots__: every instance carries a dict ----
class LogRecordDict:
    def __init__(self, ts: str, level: str, message: str) -> None:
        self.ts = ts
        self.level = level
        self.message = message


# ---- With __slots__: fixed attribute list, no per-instance dict ----
class LogRecordSlots:
    __slots__ = ("ts", "level", "message")

    def __init__(self, ts: str, level: str, message: str) -> None:
        self.ts = ts
        self.level = level
        self.message = message


a = LogRecordDict("2024-01-01", "ERROR", "db timeout")
b = LogRecordSlots("2024-01-01", "ERROR", "db timeout")

size_a = sys.getsizeof(a) + sys.getsizeof(a.__dict__)
size_b = sys.getsizeof(b)

print(f"without __slots__: {size_a:>4} bytes  (object + __dict__)")
print(f"with    __slots__: {size_b:>4} bytes")
print(f"saving per instance: {size_a - size_b} bytes")
print(f"over 1,000,000 records: ~{(size_a - size_b) * 1_000_000 / 1024 / 1024:.0f} MB")

# ---- The second benefit: typos become errors ----
a.mesage = "typo"                    # silently creates a NEW attribute
print("\ntypo on dict version   :", a.mesage, " <- accepted, bug is now invisible")

try:
    b.mesage = "typo"
except AttributeError as exc:
    print("typo on slots version  :", exc)

# ---- The costs ----
print("\nhas __dict__?  dict version:", hasattr(a, "__dict__"),
      "| slots version:", hasattr(b, "__dict__"))

print("""
Costs of __slots__:
  - No __dict__, so no adding attributes at run time
  - Subclasses need their own __slots__ or they regain a __dict__
  - Does not play well with multiple inheritance
  - Cannot have class-level defaults for slotted names

Use it when: many instances (100k+) AND memory profiling says so.
Do NOT use it: by default, "just in case", or on classes with a handful of instances.
""")

# @dataclass(slots=True) does this for you - see 5.3

---

## Dunder methods: a reference

"Dunder" = **d**ouble **under**score. These are the hooks Python calls on your behalf when
a built-in operation touches your object. Implementing them is how a class stops being a
bag of data and starts behaving like a native Python type.

| Group | Methods | Triggered by |
|---|---|---|
| **Construction** | `__new__`, `__init__`, `__del__` | creating / destroying |
| **Representation** | `__repr__`, `__str__`, `__format__` | `repr()`, `print()`, f-strings |
| **Comparison** | `__eq__`, `__lt__`, `__le__`, `__gt__`, `__ge__`, `__ne__` | `==`, `<`, sorting |
| **Hashing** | `__hash__` | `set`, `dict` keys |
| **Arithmetic** | `__add__`, `__sub__`, `__mul__`, `__truediv__` | `+`, `-`, `*`, `/` |
| **Container** | `__len__`, `__getitem__`, `__setitem__`, `__delitem__`, `__contains__` | `len()`, `x[k]`, `in` |
| **Iteration** | `__iter__`, `__next__` | `for`, unpacking |
| **Callable** | `__call__` | `obj()` |
| **Context manager** | `__enter__`, `__exit__` | `with` — see **06** |
| **Attribute access** | `__getattr__`, `__setattr__`, `__getattribute__` | `obj.x` |

**Real-world use case:** implementing `__len__`, `__getitem__` and `__iter__` is what makes
a custom collection work with `for`, `len()`, slicing, comprehensions and unpacking — with
no extra effort from whoever uses it.

In [ ]:
# The dunder methods you have met, and where each one is triggered.
class Repository:
    """A tiny in-memory repository, implementing the container protocol."""

    def __init__(self, name: str) -> None:
        self.name = name
        self._items: dict[int, str] = {}

    # --- representation ---
    def __repr__(self) -> str:
        return f"Repository(name={self.name!r}, items={len(self._items)})"

    # --- sized / container / iterable ---
    def __len__(self) -> int:
        return len(self._items)

    def __contains__(self, key: int) -> bool:
        return key in self._items

    def __iter__(self):
        return iter(self._items.values())

    # --- subscripting ---
    def __getitem__(self, key: int) -> str:
        return self._items[key]

    def __setitem__(self, key: int, value: str) -> None:
        self._items[key] = value

    def __delitem__(self, key: int) -> None:
        del self._items[key]

    # --- truthiness: empty repository is falsy ---
    def __bool__(self) -> bool:
        return bool(self._items)


repo = Repository("users")
print("empty, truthy? ", bool(repo))

repo[1] = "aditya"          # __setitem__
repo[2] = "priya"

print("repr           :", repr(repo))
print("len()          :", len(repo))          # __len__
print("repo[1]        :", repo[1])            # __getitem__
print("2 in repo      :", 2 in repo)          # __contains__
print("list(repo)     :", list(repo))         # __iter__
print("bool(repo)     :", bool(repo))         # __bool__

del repo[1]                                    # __delitem__
print("after delete   :", repo)

# Because __iter__ and __len__ exist, the object works with the whole language:
print("\nsorted()       :", sorted(repo))
print("comprehension  :", [name.title() for name in repo])
print("unpacking      :", [*repo])

*Before the wrap-up, one appendix: the underscore has appeared throughout this notebook (`_protected`, `__private`, `__init__`) - here are all its meanings collected in one place.*

## Underscore (_) in Python
> The underscore (_) is special in Python.


### Case 1: Storing the value of the last expression in the interpreter
- The interactive interpreter (REPL) stores the last expression value in the special variable `_`:

```python
>>> 45 + 5
50
>>> _          # value of the last expression
50
>>> _ * 2
100
```


### Case 2: Ignoring values
- If a value must be unpacked but is not needed, just assign it to underscore:

```python
status, _ = ('200 OK', b'payload we do not need')    # ignore one value
code, _, _ = (500, 'Internal Server Error', None)    # ignore several values

for _ in range(3):        # loop where the counter itself is never used
    print('retrying request...')
```


### Case 3: Special meanings in names of variables and functions

#### _single_leading_underscore
- This convention is used for declaring **internal** ("protected") variables, functions, methods and classes in a module.
- Anything with this convention is skipped by a wildcard import:
```python
from module import *
```
>However, if you still need to import an internal name, import it directly.


#### single_trailing_underscore_
- This convention can be used for avoiding conflict with Python keywords or built-ins.
- For example, to avoid conflict with the 'list' built-in type:
```python
list_ = [1, 2, 3, 4, 5]
```


#### __double_leading_underscore
- A double leading underscore triggers **name mangling** inside a class body, which changes how the attribute is reached from outside.
- We cannot access such methods/attributes with the plain syntax **`obj.__method`**:

```python
class ApiClient:
    def __refresh_token(self):   # stored as _ApiClient__refresh_token
        return 'new-token'

client = ApiClient()
# client.__refresh_token()       # AttributeError: no attribute '__refresh_token'
```

- We can access them with the syntax **`obj._ClassName__method`**:

```python
client._ApiClient__refresh_token()   # works: returns 'new-token'
```

- This naming convention is useful with inheritance when parent and child each need their **own** method of the same name:

```python
class Parser:
    def __validate(self):            # becomes _Parser__validate
        return 'generic checks'
    def run(self):
        return self.__validate()     # always resolves to Parser's own version

class JsonParser(Parser):
    def __validate(self):            # becomes _JsonParser__validate - a different name
        return 'json-specific checks'

print(JsonParser().run())            # 'generic checks' - the child did not clobber it
```


#### __double_leading_and_trailing_underscore__
- This convention is used for special variables or methods (so-called "magic" or **dunder** methods) such as `__init__`, `__len__`, etc. These methods provide special syntactic features or do special things - see the dunder reference table above.

In [ ]:
class A:
    def __init__(self):
        print("Object initialized!")

    def __len__(self):
        print("Get object length!")
        return 10

    def __call__(self):
        print("Object used as a function!")

    def __eq__(self, a):
        print("Equate the two objects!")

a = A()  # call __init__
print(len(a))  # call __len__
a()  # call __call__
b = A()  # call __init__
a==b  # call __eq__

### Case 4: To separate the digits of number literal value
- It is used for separating digits of numbers using underscore for readability.

In [ ]:
num = 1_000_000
print(num)

---

## Common Mistakes & Pitfalls

1. 🔴 **Defining `__eq__` without `__hash__`.** Python sets `__hash__ = None`, and your objects silently stop working as dict keys or set members. Either add `__hash__` or use `@dataclass(frozen=True)`.
2. 🔴 **Only defining `__str__`.** Debuggers, REPLs, logs and containers all use `__repr__`. Define `__repr__` first; add `__str__` only if end users see the object.
3. **Mutable class variables.** `class Deployment: tags = []` gives *every* instance the same list — the leak you watched happen in the class-variable section. Assign mutable state in `__init__` (same trap as **2.7**'s mutable default).
4. **Forgetting `super().__init__()`** in a subclass — the parent's attributes never get set, and the failure surfaces later as an `AttributeError` far from the cause.
5. **Thinking `__private` is enforced.** It is name mangling (`_Class__private`), a convention against accidents, not a security boundary — and it hides base-class attributes from subclasses in ways that surprise (`_TestGateway__api_key`).
6. **Adding `@property` getters and setters for every attribute.** In Python a plain attribute is fine until you actually need logic — you can add `@property` later without breaking callers. That is the whole point.
7. **Hand-calling each parent's `__init__` in multiple inheritance.** The last call silently wins every shared attribute, and shared grandparents can run twice. Cooperative `super()` plus a look at `__mro__` is the honest fix.
8. **Using a class where a function would do.** A class with one method and no state is a function wearing a costume.
9. **Deep inheritance chains.** Past two or three levels, the MRO becomes hard to reason about. Prefer composition — "has a" beats "is a" whenever it is the truer sentence (see **5.4**).
10. **Relying on `__del__` for cleanup.** Its timing is not guaranteed. Use a context manager (**06 Exception Handling**).

## Best Practices

- Define **`__repr__`** on every class you will ever debug. Make it unambiguous.
- Define `__eq__` and `__hash__` **together**, over the same fields.
- Use `@dataclass` when the class is mostly data — it writes these for you (see **5.3**).
- Start with plain attributes; introduce `@property` only when you need validation or a computed value.
- Use `_single_underscore` for internal attributes; reserve `__double` for genuine name-collision avoidance in subclasses.
- Give alternative construction paths a `from_*` class method each (`Job.from_line`, `Job.from_dict`) instead of one `__init__` full of flags.
- Prefer **composition over inheritance** — inject the varying part (a `RetryPolicy`, a `StorageBackend`) rather than subclassing for every combination.
- Use `super()` with no arguments; it follows the MRO and stays correct under multiple inheritance.
- Make abstract contracts explicit with `ABC` + `@abstractmethod` — a half-implemented backend should fail at construction, not return `None` at 2am.
- Annotate class attributes and method signatures (see **4.5**); return `Self` from alternative constructors (**16.4**).
- Add `__slots__` only when profiling says memory matters, and you have many instances.

## Practice Exercises

Try these before moving on.

1. Write a `Money` class with `__repr__`, `__eq__`, `__hash__` and `__add__`, refusing to add different currencies (return `NotImplemented` for non-`Money` operands; raise `ValueError` for currency mismatches). Add `__radd__` so `sum()` works on a list of prices.
2. Give a `Version` class `__eq__` and `__lt__`, add `@total_ordering`, and sort a list of releases.
3. Extend the notifier family: write a `WebhookNotifier(Notifier)` that POSTs (pretend: print) JSON, then a `ThrottledWebhookNotifier` on top of it that drops every message after the first three per run — decide at each level whether to call `super().send()`.
4. Build a `RateLimiter` class with a class variable tracking global calls and an instance variable tracking per-key calls. Show why the class variable must not be a mutable default.
5. Write a `JobConfig` class where `timeout` is a `@property` that rejects non-positive values, and `retries_left` is a computed read-only property.
6. Measure the memory of 100,000 instances with and without `__slots__` using `sys.getsizeof`.
7. Write a `ConnectionPool` that returns the same instance every time, using `__new__`.
8. Demonstrate `__eq__`-without-`__hash__` breaking a `set`, then fix it.
9. Turn `RetrySchedule` into an iterator that adds ±10% jitter (use `random`) and caps each delay at a `max_delay` — keep `StopIteration` semantics intact.
10. Define a `StorageBackend` ABC with `save`/`load`/`delete`, implement `InMemoryStorage` and a `CountingStorage` wrapper that **composes** any backend and counts calls — no inheritance between the two concrete classes.